In [1]:
%pip install keras
%pip install tensorflow

In [2]:


import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


In [4]:
import numpy as np

data_path = '.'
fname = 'timewise_5s_500p'

X = np.load(f'{data_path}/{fname}_X.npy')  # shape: (N, T)
y = np.load(f'{data_path}/{fname}_y.npy')  # shape: (N,)

print(X.shape, y.shape)

shuffle(X, y, random_state=123)

print(np.bincount(y))


(26088, 500) (26088,)
[  0 383 287 224 308 320 174 297 353 276 427 176 196 296 454 386 360 193
 443 388 408 402 392 382 399 339 287 331 280 110 414 346 276 292 290 181
 184 213 393 308 204 226 269 267 199 123 380  87  94 443 123 457 407 385
 371 202 351 168 307 171 268 183 360 166 284 285 175 374 281 239  87 404
  89   0 214 277 328 172 368 400 342 329 406 291 174 173 279 317 293 195
 265 214 154 300]


In [5]:
from sklearn.preprocessing import LabelEncoder
# Fit on the full set of labels (before train/test split)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Then split the encoded labels
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [6]:
scaler = StandardScaler()

# Reshape for scaler: (N, T) → (N*T,)
X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
X_val_reshaped = X_val.reshape(X_val.shape[0], -1)
X_test_reshaped = X_test.reshape(X_test.shape[0], -1)

# Fit on training only
scaler.fit(X_train_reshaped)

X_train_scaled = scaler.transform(X_train_reshaped)
X_val_scaled = scaler.transform(X_val_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)


In [7]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_encoded))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [18]:
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_val_seq   = X_val_scaled.reshape((X_val_scaled.shape[0], X_val_scaled.shape[1], 1))
X_test_seq  = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))


In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, BatchNormalization, GlobalAveragePooling1D

input_shape = (X_train_seq.shape[1], 1)  # (time_steps, features)

inputs = Input(shape=input_shape)

# 🔍 Feature extractor
x = Conv1D(64, kernel_size=5, activation='relu', padding='same')(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.2)(x)

x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.2)(x)

# 🔁 Sequence modeler
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dropout(0.3)(x)

# 🧠 Classifier head
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



In [10]:
print(X_train_seq.shape)  # should be (samples, time_steps, features)
print(np.std(X_train_seq))  # should not be zero or near-zero


(18261, 500, 1)
1.0000000000000002


In [11]:
print(y_train_cat.shape)  # should be (samples, num_classes)
print(np.unique(np.argmax(y_train_cat, axis=1)))  # Should cover multiple labels


(18261, 92)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91]


In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=40,
    batch_size=32,
    callbacks = [early_stop]
)


Epoch 1/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.0185 - loss: 4.4448 - val_accuracy: 0.0409 - val_loss: 4.2244
Epoch 2/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 17s 19ms/step - accuracy: 0.0460 - loss: 4.1325 - val_accuracy: 0.0979 - val_loss: 3.8212
Epoch 3/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.0843 - loss: 3.8322 - val_accuracy: 0.1510 - val_loss: 3.3994
Epoch 4/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.1337 - loss: 3.4863 - val_accuracy: 0.1845 - val_loss: 3.1219
Epoch 5/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.1636 - loss: 3.2778 - val_accuracy: 0.2259 - val_loss: 3.0035
Epoch 6/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.2017 - loss: 3.0408 - val_accuracy: 0.2880 - val_loss: 2.6762
Epoch 7/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.2382 - loss: 2.8597 - val_accuracy: 0.3470 - val_loss: 2.4155
Epoch 8/40
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.2846 - loss: 2.6399 - 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense, Input

model2 = Sequential([
    Input(shape=(X_train_seq.shape[1], 1)),

    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    LSTM(64),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])


In [30]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D,
    Add, Embedding
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- CONFIG ---
embed_dim = 16
num_heads = 2
ff_dim = 32
dropout = 0.3
seq_len = X_train_seq.shape[1]
input_shape = (seq_len, 1)

# --- Transformer Encoder Block ---
def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout=0.1):
    # Multi-head self-attention
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    attention_output = Dropout(dropout)(attention_output)
    out1 = LayerNormalization(epsilon=1e-6)(inputs + attention_output)

    # Feedforward network
    ffn_output = Dense(ff_dim, activation="relu")(out1)
    ffn_output = Dense(embed_dim)(ffn_output)
    ffn_output = Dropout(dropout)(ffn_output)
    return LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

# --- Inputs & Positional Embedding ---
input_layer = Input(shape=input_shape)
x = Dense(embed_dim)(input_layer)  # Project to embedding dimension

# Learnable positional embeddings
position_indices = tf.range(start=0, limit=seq_len, delta=1)
position_embed_layer = Embedding(input_dim=seq_len, output_dim=embed_dim)
position_embeddings = position_embed_layer(position_indices)
# Reshape positional embeddings to (1, seq_len, embed_dim) for broadcasting
position_embeddings = tf.expand_dims(position_embeddings, axis=0)

x = Add()([x, position_embeddings])

# --- Transformer ---
x = transformer_encoder(x, embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)

# --- Classification head ---
x = GlobalAveragePooling1D()(x)
x = Dropout(dropout)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(dropout)(x)
output = Dense(num_classes, activation="softmax")(x)

model3 = Model(inputs=input_layer, outputs=output)

# --- Compile ---
model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])




In [ ]:
# --- Callbacks ---
checkpoint = ModelCheckpoint("best_transformer_model.keras", save_best_only=True, monitor="val_loss", mode="min", verbose=1)
early_stop = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)

# --- Train ---
model3.fit(
    X_train_seq, y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=50,
    batch_size=32,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

Epoch 1/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0145 - loss: 4.4966
Epoch 1: val_loss improved from inf to 4.29230, saving model to best_transformer_model.keras
571/571 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.0145 - loss: 4.4966 - val_accuracy: 0.0299 - val_loss: 4.2923
Epoch 2/50
568/571 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0304 - loss: 4.2799
Epoch 2: val_loss improved from 4.29230 to 4.08355, saving model to best_transformer_model.keras
571/571 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.0304 - loss: 4.2797 - val_accuracy: 0.0544 - val_loss: 4.0835
Epoch 3/50
571/571 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0537 - loss: 4.1178
Epoch 3: val_loss improved from 4.08355 to 3.88904, saving model to best_transformer_model.keras
571/571 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.0537 - loss: 4.1178 - val_accuracy: 0.0754 - val_loss: 3.8890
Epoch 4/50
569/571 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0585 - loss: 3.9640
Epoch 4: val

In [19]:
model.save("./model-cnn-lstm.keras")




ValueError: as_list() is not defined on an unknown TensorShape.

In [20]:
print(X_test_seq.shape)

(3914, 500, 1)


In [21]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=1)
print(f"✅ Test accuracy: {test_acc:.2%}")

123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8219 - loss: 0.6289
✅ Test accuracy: 83.06%
